In [6]:
import pandas as pd
import pycountry
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from ipywidgets import VBox, Button, HTML as IpyHTML

# 读取 parquet 数据
df = pd.read_parquet("data/AllMeds_dataset.parquet")

# 识别国家字段
country_col = "country" if "country" in df.columns else "Country"

# 统计每个国家的数据量
country_counts = (
    df[country_col]
    .dropna()
    .astype(str)
    .str.strip()
    .replace("", pd.NA)
    .dropna()
    .value_counts()
    .rename_axis("Country")
    .reset_index(name="record_count")
)


def get_country_name(value):
    if pd.isna(value):
        return None
    name = str(value).strip()
    if not name:
        return None
    for candidate in [name, name.upper(), name.title()]:
        try:
            country_obj = pycountry.countries.lookup(candidate)
            return getattr(country_obj, "common_name", None) or getattr(country_obj, "name", None)
        except LookupError:
            continue
    return name


def get_language_name(value):
    if pd.isna(value):
        return None
    name = str(value).strip()
    if not name:
        return None
    for candidate in [name, name.upper(), name.title()]:
        try:
            return pycountry.languages.lookup(candidate).name
        except LookupError:
            continue
    return name


def get_iso3(value):
    if pd.isna(value):
        return None

    name = str(value).strip()
    if not name:
        return None

    for candidate in [name.upper(), name.title(), name]:
        try:
            return pycountry.countries.lookup(candidate).alpha_3
        except LookupError:
            continue

    aliases = {
        "United States": "United States of America",
        "USA": "United States of America",
        "Russia": "Russian Federation",
        "South Korea": "Korea, Republic of",
        "The Netherlands": "Netherlands",
        "Czechia": "Czech Republic",
        "Taiwan": "Taiwan, Province of China",
        "Vietnam": "Socialist Republic of Viet Nam",
        "Palestine": "the State of Palestine",
        "Moldova": "Republic of Moldova",
        "Bolivia": "Bolivia, Plurinational State of",
    }

    alias_name = aliases.get(name)
    if alias_name:
        try:
            return pycountry.countries.lookup(alias_name).alpha_3
        except LookupError:
            return None

    return None


country_counts["Country_Name"] = country_counts["Country"].apply(get_country_name)
country_counts["ISO3"] = country_counts["Country"].apply(get_iso3)
country_counts = country_counts.dropna(subset=["ISO3"]).copy()
country_counts["ISO3"] = country_counts["ISO3"].str.upper()

# 关联 region / subregion
# 注意：这里不能预先创建 "lang" 列，否则下面 merge 时会和 lang_info 的
# "lang" 列同名冲突，被 pandas 自动重命名为 lang_x/lang_y，
# 导致合并后找不到 "lang" 列（这是之前报错的根本原因）。
for col in ["Region", "Subregion", "Language_Name", "lang_list"]:
    if col not in country_counts.columns:
        country_counts[col] = pd.NA

if {"lang", "country", "UN-region", "UN-sub_region"}.issubset(df.columns):
    lang_info = (
        df[["lang", "country", "UN-region", "UN-sub_region"]]
        .dropna()
        .drop_duplicates()
        .rename(columns={"country": "Country"})
    )
    lang_info["Country"] = lang_info["Country"].astype(str).str.strip()
    # 一个国家可能对应多种语言，合并成一行，用逗号连接
    lang_info = (
        lang_info.groupby("Country")
        .agg(
            lang=("lang", lambda s: ", ".join(sorted(set(s)))),
            **{
                "UN-region": ("UN-region", "first"),
                "UN-sub_region": ("UN-sub_region", "first"),
            },
        )
        .reset_index()
    )
    country_counts = country_counts.merge(lang_info, on="Country", how="left")
    country_counts["Region"] = country_counts["UN-region"].fillna("N/A")
    country_counts["Subregion"] = country_counts["UN-sub_region"].fillna("N/A")

# 统一处理语言字段，改为每个国家列出所有 lang
if "lang" in country_counts.columns:
    country_counts["lang"] = country_counts["lang"].fillna("N/A")
    country_counts["lang"] = country_counts["lang"].astype(str).str.strip().str.upper()
    country_counts["lang_list"] = country_counts["lang"].apply(lambda x: [item.strip().upper() for item in x.split(",") if item.strip()])
    country_counts["lang_list"] = country_counts["lang_list"].apply(lambda x: x if x else ["N/A"])
    country_counts["Language_Name"] = country_counts["lang_list"].apply(lambda x: ", ".join([get_language_name(item) for item in x if item != "N/A"]))
else:
    country_counts["lang"] = "N/A"
    country_counts["lang_list"] = pd.Series([["N/A"] for _ in range(len(country_counts))], index=country_counts.index)
    country_counts["Language_Name"] = "N/A"

# 构造 hover 文本
country_counts["hover_text"] = (
    "Country: " + country_counts["Country_Name"].fillna(country_counts["Country"]).astype(str)
    + "<br>"
    + "Language Codes: " + country_counts["lang"].fillna("N/A").astype(str).str.upper()
    + "<br>"
    + "Language Names: " + country_counts["Language_Name"].fillna("N/A").astype(str)
    + "<br>"
    + "Region: " + country_counts["Region"].fillna("N/A").astype(str)
    + "<br>"
    + "Subregion: " + country_counts["Subregion"].fillna("N/A").astype(str)
    + "<br>"
    + "Record Count: " + country_counts["record_count"].astype(str)
)

# 六档离散配色（同一深蓝色 #003366，用不同透明度表现分级）
NAVY_RGB = "0, 51, 102"  # #003366

LEVEL_BINS = [
    (10000, 1.00, "≥10,000"),
    (1000, 0.85, "1,000–9,999"),
    (100, 0.70, "100–999"),
    (10, 0.55, "10–99"),
    (1, 0.40, "1–9"),
    (0, 0.25, "<1"),
]


def assign_color_level(volume):
    for threshold, level, _ in LEVEL_BINS:
        if volume >= threshold:
            return level
    return LEVEL_BINS[-1][1]


def assign_level_label(volume):
    for threshold, _, label in LEVEL_BINS:
        if volume >= threshold:
            return label
    return LEVEL_BINS[-1][2]


country_counts["Level"] = country_counts["record_count"].apply(assign_color_level)
country_counts["Level_Label"] = country_counts["record_count"].apply(assign_level_label)

level_order = [label for _, _, label in LEVEL_BINS]
level_color_map = {label: f"rgba({NAVY_RGB}, {level})" for _, level, label in LEVEL_BINS}

base_fig = px.choropleth(
    country_counts,
    locations="ISO3",
    locationmode="ISO-3",
    color="Level_Label",
    category_orders={"Level_Label": level_order},
    color_discrete_map=level_color_map,
    hover_name="Country",
    hover_data={"record_count": True, "ISO3": False, "Level_Label": False},
    custom_data=[country_counts["hover_text"]],
    title="Number of Records per Country",
)

base_fig.update_traces(
    hovertemplate="%{customdata[0]}<extra></extra>",
)

base_fig.update_geos(
    projection_type="natural earth",
    showcountries=True,
    countrycolor="lightgray",
    showframe=False,
)

base_fig.update_layout(
    margin=dict(l=0, r=0, t=50, b=0),
    legend_title_text="Records",
)

# 转成可交互的 FigureWidget
fig = go.FigureWidget(base_fig)

# 详情面板
info_box = IpyHTML(value="<b>Select a country</b>")


# 离散配色下 px.choropleth 会按类别拆成多条 trace，
# 所以点击回调要按 trace.locations[idx] 反查行，不能像单 trace 时那样直接用 idx。
def on_click(trace, points, selector):
    if not points.point_inds:
        return
    idx = points.point_inds[0]
    iso3 = trace.locations[idx]
    row = country_counts[country_counts["ISO3"] == iso3].iloc[0]
    info_box.value = (
        f"<b>{row['Country_Name']}</b><br>"
        f"Language Codes: {row['lang'] if pd.notna(row['lang']) else 'N/A'}<br>"
        f"Language Names: {row['Language_Name'] if pd.notna(row['Language_Name']) else 'N/A'}<br>"
        f"Region: {row['Region'] if pd.notna(row['Region']) else 'N/A'}<br>"
        f"Subregion: {row['Subregion'] if pd.notna(row['Subregion']) else 'N/A'}<br>"
        f"Record Count: {row['record_count']}"
    )


for trace in fig.data:
    trace.on_click(on_click)

reset_button = Button(description="Reset Zoom")


def reset_map(_):
    info_box.value = "<b>Select a country</b>"


reset_button.on_click(reset_map)

# 展示
output = VBox([info_box, reset_button, fig])
display(output)

# 保存 HTML
output_html = "data_distribution/Interactive_Map.html"
base_fig.write_html(output_html, include_plotlyjs="cdn")
